# C4b · Timbre: MFCCs, spectral flux, and a table for statistics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juansgomez87/py-mir-toolbox/blob/main/docs/chapters/C4b_mfcc_flux_statistics.ipynb)

This chapter finishes the timbre features and then prepares them for statistics:

- **MFCCs** (mel-frequency cepstral coefficients) summarise the shape of the spectrum in 13 numbers;
- the **spectral flux** measures how much the spectrum changes from one frame to the next;
- **summary statistics** turn each feature curve into a few numbers per piece, and `mir.batch` builds a table with one row per audio file, ready for R, SPSS or JASP.

Each section has the same steps as before: ① the idea, with sound; ② the maths in NumPy; ③ mirpy; ④ librosa; ⑤ Essentia; ⑥ the three libraries side by side; and an exercise.

## Setup

In Colab, the next cell installs the libraries. On your own computer, see *Getting started*.

In [ ]:
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic(
        "pip", "install -q git+https://github.com/juansgomez87/py-mir-toolbox librosa essentia"
    )

In [ ]:
import tempfile
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.fft
import soundfile as sf
from IPython.display import Audio
from scipy.signal import sawtooth

import mirpy as mir

try:
    import essentia
    import essentia.standard as es

    essentia.log.infoActive = False  # hide Essentia's progress messages
except ImportError:
    es = None
    print("Essentia is not installed here; the Essentia cells will be skipped.")

The same three-part test sound as in C4a (one second each of a sine, a sawtooth and white noise), and the same frame settings for all libraries: 50 ms frames, with a hop of half a frame.

In [ ]:
sr = 22050
t = np.arange(sr) / sr
rng = np.random.default_rng(0)

sine = 0.5 * np.sin(2 * np.pi * 440 * t)
saw = 0.5 * sawtooth(2 * np.pi * 440 * t)
noise = rng.uniform(-0.5, 0.5, t.size)  # white noise, never beyond ±0.5
three_parts = np.concatenate([sine, saw, noise])
a = mir.audio(three_parts, sr=sr)

frame_length = int(0.05 * sr)  # 1102 samples
hop = int(0.025 * sr)  # 551 samples (librosa and Essentia need a whole number)
n_fft = 2048

Audio(three_parts, rate=sr, normalize=False)

In [ ]:
def essentia_spectra(x):
    """Magnitude spectrum of each frame with Essentia (see C4a), one row per frame."""
    window = es.Windowing(
        type="hamming", size=frame_length, zeroPadding=n_fft - frame_length, normalized=False
    )
    spectrum = es.Spectrum(size=n_fft)
    frames = es.FrameGenerator(
        x.astype(np.float32), frameSize=frame_length, hopSize=hop, startFromZero=True
    )
    return np.array([spectrum(window(frame)) for frame in frames])


def essentia_times(n_frames):
    """Centre time of each Essentia frame (first frame starting at 0)."""
    return (np.arange(n_frames) * hop + frame_length / 2) / sr


def colour_limit(values):
    """Symmetric colour limit that ignores the most extreme 2 % of values."""
    return np.percentile(np.abs(values), 98)


def part_lines(ax, borders=(1, 2)):
    """Mark the borders between the parts of a test sound."""
    for border in borders:
        ax.axvline(border, color="lightgray", linewidth=1)

## 1. MFCCs

### ① Idea

The spectrum of a frame has about a thousand values. **MFCCs** describe its overall shape, its *envelope*, with only 13 numbers. They ignore the exact position of the harmonics (the pitch) and keep the broad outline (much of the timbre). That is why they are among the most used features in speech and music research.

MIRtoolbox computes them in three steps:

1. group the spectrum into 40 **mel bands**. The mel scale follows our hearing: bands are narrow at low frequencies and wide at high ones;
2. take the **logarithm** of each band, since we hear loudness on a roughly logarithmic scale;
3. apply a **discrete cosine transform** (DCT). It describes the 40 band values as a sum of cosine shapes: coefficient 1 is the overall tilt (more low or more high frequencies), coefficient 2 a curve, and higher coefficients finer and finer detail. MIRtoolbox keeps coefficients 1 to 13.

### ② From scratch

Writing the mel bands by hand takes many lines, so we take steps 1 and 2 from `mir.spectrum` (`band="mel", log=True`) and do the DCT ourselves with SciPy.

In [ ]:
mel = mir.spectrum(a, frame=True, band="mel", n_bands=40, log=True)
mfcc_from_scratch = scipy.fft.dct(mel.data[0], type=2, norm="ortho", axis=0)[1:14]

frame_index = 60  # a frame in the sawtooth part
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(mel.x_values, mel.data[0, :, frame_index], marker=".")
axes[0].set_xlabel("Mel band")
axes[0].set_title("1–2. mel bands, log scale")
axes[1].bar(np.arange(1, 14), mfcc_from_scratch[:, frame_index])
axes[1].set_xlabel("Coefficient")
axes[1].set_title("3. DCT: MFCCs 1–13")
plt.show()

### ③ mirpy

`mir.mfcc` is MIRtoolbox `mirmfcc`. The options are `bands` (MATLAB `'Bands'`, default 40), `rank` (MATLAB `'Rank'`, default 1 to 13; 0 is the average energy) and `delta` (MATLAB `'Delta'`), which gives the change of each coefficient over time.

Over all frames, the MFCCs make an image like a spectrogram, with the 13 coefficients on the vertical axis.

In [ ]:
mfcc = mir.mfcc(a, frame=True)
print("same as our NumPy version:", np.allclose(mfcc.data[0], mfcc_from_scratch))

fig, ax = plt.subplots(figsize=(9, 3))
limit = colour_limit(mfcc.data[0])
image = ax.pcolormesh(
    mfcc.frame_centers,
    mfcc.x_values,
    mfcc.data[0],
    shading="auto",
    cmap="coolwarm",
    vmin=-limit,
    vmax=limit,
)
fig.colorbar(image, ax=ax)
part_lines(ax)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Coefficient")
ax.set_title("mirpy MFCCs: sine | sawtooth | noise")
plt.show()

### ④ librosa

`librosa.feature.mfcc` follows the same three steps, with its own choices: 128 mel bands by default (we set 40), the log of the **power** in decibels, and coefficient 0 included. We ask for 14 coefficients and drop the first.

In [ ]:
librosa_mfcc = librosa.feature.mfcc(
    y=three_parts,
    sr=sr,
    n_mfcc=14,
    n_mels=40,
    n_fft=n_fft,
    hop_length=hop,
    win_length=frame_length,
    window="hamming",
)[1:]
librosa_times = librosa.times_like(librosa_mfcc, sr=sr, hop_length=hop)
print(librosa_mfcc.shape)  # (coefficients, frames)

### ⑤ Essentia

Essentia's `MFCC` algorithm takes a magnitude spectrum and returns the mel bands and the coefficients. Its log is in decibels too, and coefficient 0 is included.

In [ ]:
def essentia_mfcc(spectra):
    """MFCCs 1–13 of each Essentia spectrum, one column per frame."""
    mfcc_algorithm = es.MFCC(
        inputSize=n_fft // 2 + 1,
        sampleRate=sr,
        numberBands=40,
        numberCoefficients=14,
        highFrequencyBound=sr / 2,
    )
    return np.array([mfcc_algorithm(m)[1][1:] for m in spectra]).T


if es is None:
    print("Essentia is not available; skipped.")
else:
    essentia_mfcc_values = essentia_mfcc(essentia_spectra(three_parts))
    print(essentia_mfcc_values.shape)  # (coefficients, frames)

### ⑥ Side by side

The three MFCC images, each with its own colour scale:

In [ ]:
images = [("mirpy", mfcc.frame_centers, mfcc.data[0]), ("librosa", librosa_times, librosa_mfcc)]
if es is not None:
    images.append(
        ("Essentia", essentia_times(essentia_mfcc_values.shape[1]), essentia_mfcc_values)
    )

fig, axes = plt.subplots(len(images), 1, figsize=(9, 2.3 * len(images)), sharex=True)
for ax, (name, times, values) in zip(axes, images, strict=True):
    limit = colour_limit(values)
    image = ax.pcolormesh(
        times, np.arange(1, 14), values, shading="auto", cmap="coolwarm", vmin=-limit, vmax=limit
    )
    fig.colorbar(image, ax=ax)
    part_lines(ax)
    ax.set_ylabel("Coefficient")
    ax.set_title(name, loc="left")
axes[-1].set_xlabel("Time (s)")
plt.show()

- All three show the same three blocks: the sine, the sawtooth and the noise each have their own MFCC pattern.
- The values are on different scales. **mirpy** uses the log of the magnitude (as MIRtoolbox); **librosa** uses decibels of the power; **Essentia** uses decibels of the magnitude. The mel bands are also drawn a little differently.
- So MFCCs from different libraries cannot be mixed in one analysis. Pick one library and use it for all your files.

### Exercise

Average the MFCCs of each part (sine, sawtooth, noise) over its frames and draw the three averages as bar charts. Which coefficients separate the three sounds most clearly?

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
parts = {"sine": (0, 1), "sawtooth": (1, 2), "noise": (2, 3)}
fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, (name, (start, end)) in zip(axes, parts.items(), strict=True):
    inside = (mfcc.frame_centers > start) & (mfcc.frame_centers < end)
    ax.bar(np.arange(1, 14), mfcc.data[0][:, inside].mean(axis=1))
    ax.set_title(name)
    ax.set_xlabel("Coefficient")
plt.show()

The sine has large low coefficients: its spectrum is a single peak at the bottom, a strong tilt. The sawtooth has large high coefficients, which describe the fine, regular ripple of its harmonics. The noise has values close to 0: a flat spectrum has no shape to describe.

## 2. Spectral flux

### ① Idea

The **spectral flux** measures how much the spectrum changes between one frame and the next. During a held note, the spectrum stays the same and the flux is low. When a new note starts, the spectrum changes and the flux jumps. Its peaks mark moments of contrast in the sound, such as note onsets.

A test sound: a C major scale played with sawtooth notes, a quarter of a second each.

In [ ]:
scale = [262, 294, 330, 349, 392, 440, 494, 523]  # C4 to C5, in Hz
note = t[: sr // 4]
melody = np.concatenate([0.4 * sawtooth(2 * np.pi * f0 * note) for f0 in scale])
onsets = np.arange(1, len(scale)) * 0.25  # times when a new note starts

Audio(melody, rate=sr)

### ② From scratch

Compute the spectrum of every frame (as in C4a), then the distance between each spectrum and the next. MIRtoolbox uses the **Euclidean** distance: the square root of the sum of the squared differences.

In [ ]:
n_frames = (melody.size - frame_length) // hop + 1
frames = melody[np.arange(n_frames)[:, np.newaxis] * hop + np.arange(frame_length)]
spectra = np.abs(np.fft.rfft(frames * np.hamming(frame_length), n=n_fft, axis=1))
flux_from_scratch = np.sqrt(np.sum(np.diff(spectra, axis=0) ** 2, axis=1))
flux_times = (np.arange(1, n_frames) * hop + frame_length / 2) / sr  # at the second frame

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(flux_times, flux_from_scratch)
part_lines(ax, onsets)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Flux")
ax.set_title("Flux from scratch (gray lines: note onsets)")
plt.show()

### ③ mirpy

`mir.flux` is MIRtoolbox `mirflux`. It frames the sound by default. Some useful options:

- `dist`: the distance between frames (MATLAB `'Dist'`): `"euclidean"` (default), `"cityblock"` or `"cosine"`;
- `inc=True`: count only increases of energy, which focuses on onsets (MATLAB `'Inc'`);
- `median=True`: subtract a moving median, which removes small peaks (MATLAB `'Median'`).

In [ ]:
m = mir.audio(melody, sr=sr)
flux = mir.flux(m)
flux_inc = mir.flux(m, inc=True)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(flux.frame_centers, flux.data[0, 0], label="flux")
ax.plot(flux_inc.frame_centers, flux_inc.data[0, 0], label="inc=True (increases only)")
part_lines(ax, onsets)
ax.set_xlabel("Time (s)")
ax.legend()
plt.show()

### ④ librosa

librosa has no spectral flux function by that name. The plain flux is one line on `librosa.stft`.

In [ ]:
melody_stft = np.abs(
    librosa.stft(melody, n_fft=n_fft, hop_length=hop, win_length=frame_length, window="hamming")
)
librosa_flux = np.sqrt(np.sum(np.diff(melody_stft, axis=1) ** 2, axis=0))
librosa_flux_times = librosa.times_like(melody_stft, sr=sr, hop_length=hop)[1:]

### ⑤ Essentia

Essentia's `Flux` algorithm remembers the previous spectrum and returns the distance to it. Its first value compares the first frame with silence, so it is large; we leave it out.

In [ ]:
if es is None:
    print("Essentia is not available; skipped.")
else:
    flux_algorithm = es.Flux()
    essentia_flux = np.array([flux_algorithm(m) for m in essentia_spectra(melody)])[1:]
    essentia_flux_times = essentia_times(essentia_flux.size + 1)[1:]

### ⑥ Side by side

The curves have different units, so each is scaled so that its highest peak is 1.

In [ ]:
def scaled(values):
    """Divide a curve by its maximum, to compare shapes."""
    return values / np.max(values)


fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(flux.frame_centers, scaled(flux.data[0, 0]), linewidth=3, alpha=0.5, label="mirpy")
ax.plot(librosa_flux_times, scaled(librosa_flux), label="librosa (from stft)")
if es is not None:
    ax.plot(essentia_flux_times, scaled(essentia_flux), "--", label="Essentia")
part_lines(ax, onsets)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Scaled flux")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

- **mirpy**, **Essentia** and the one-line **librosa** flux compute the same distance between successive spectra, and their peaks fall on the note onsets.
- The one-line **librosa** flux also peaks at the very start and end, where its centred frames are half padding (C3).
- For finding note onsets, librosa has `librosa.onset.onset_strength`. It is a flux too, but on mel bands in decibels, and it keeps only increases, like `mir.flux(..., inc=True)`.

### Exercise

Alluri and Toiviainen (2010, 2012) computed the flux separately in 10 frequency bands, with 25 ms frames. They linked the flux around 50–200 Hz (bands 2 and 3) to perceived *fullness*, and around 1600–6400 Hz (bands 7 and 8) to perceived *activity*. In mirpy this is `mir.flux(x, sub_band="manual", frame=(0.025, 0.5))` (MATLAB `'SubBand'`).

Compute it for the first 15 seconds of Kevin MacLeod's "Vibe Ace" (CC BY 4.0, from librosa's examples) and plot the flux of bands 2–3 and 7–8 over time. (This file has a sampling rate of 22 050 Hz, so it has no frequencies above 11 025 Hz; the top band, above 12 800 Hz, is dropped and 9 bands remain.)

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
vibe = mir.audio(librosa.ex("vibeace"), extract=(0, 15))
band_flux = mir.flux(vibe, sub_band="manual", frame=(0.025, 0.5))
print(band_flux.data.shape)  # (bands, 1, frames): 9 bands at 22 050 Hz

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(band_flux.frame_centers, band_flux.data[1:3, 0].mean(axis=0), label="bands 2–3 (fullness)")
ax.plot(band_flux.frame_centers, band_flux.data[6:8, 0].mean(axis=0), label="bands 7–8 (activity)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Flux")
ax.legend()
plt.show()

## 3. From curves to a table for statistics

### ① Idea

A feature curve has one value per frame: thousands of values for a song. For statistics, such as comparing pieces or relating features to listeners' ratings, we usually want **one row per piece** with a few numbers per feature. MIRtoolbox's `mirstat` gives these summaries:

- **Mean** and **Std** (standard deviation): the average level and how much it varies;
- **Slope**: whether the feature tends to rise or fall over the excerpt;
- **PeriodFreq**, **PeriodAmp** and **PeriodEntropy**: whether the curve repeats regularly, how fast, and how clearly.

`mirstat` leaves out frames without a value (NaN). We use four freely licensed recordings from librosa's examples, 30 seconds each:

In [ ]:
clips = {
    "brahms": "Brahms, Hungarian Dance No. 5; US Army Strings (public domain)",
    "nutcracker": "Tchaikovsky, Dance of the Sugar Plum Fairy; arr. Kevin MacLeod (CC BY 4.0)",
    "trumpet": "Mihai Sorohan, solo trumpet 06 (CC BY 4.0)",
    "vibeace": "Kevin MacLeod, Vibe Ace (CC BY 4.0)",
}
for name, credit in clips.items():
    print(f"{name:>10}: {credit}")

In [ ]:
trumpet = mir.audio(librosa.ex("trumpet"))
Audio(trumpet.data[0, :, 0], rate=trumpet.sr)

### ② From scratch

Mean, standard deviation and slope of the trumpet's centroid curve. For the slope, MIRtoolbox first rescales the time axis of all frames to run from −0.5 to 0.5, drops the frames without a value, rescales the curve (mean 0, standard deviation 1), and fits a straight line through 0.

In [ ]:
centroid = mir.centroid(trumpet, frame=True)
values = centroid.data[0, 0]
keep = ~np.isnan(values)  # frames with a value
times = centroid.frame_centers[keep]
curve = values[keep]

scaled_time = np.linspace(-0.5, 0.5, values.size)[keep]
scaled_curve = (curve - curve.mean()) / curve.std(ddof=1)
slope = np.linalg.lstsq(scaled_time[:, np.newaxis], scaled_curve, rcond=None)[0][0]

print(f"mean {curve.mean():.0f} Hz, std {curve.std(ddof=1):.0f} Hz, slope {slope:.2f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(times, curve, color="gray")
ax.axhline(curve.mean(), color="C0", label="mean")
ax.axhspan(
    curve.mean() - curve.std(ddof=1), curve.mean() + curve.std(ddof=1), alpha=0.15, label="± std"
)
ax.plot(times, curve.mean() + slope * scaled_time * curve.std(ddof=1), color="C3", label="slope")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Centroid (Hz)")
ax.legend()
plt.show()

### ③ mirpy

`mir.stat` is MIRtoolbox `mirstat`. It returns a dictionary with one entry per summary.

In [ ]:
summary = mir.stat(centroid)
for field, value in summary.items():
    print(f"{field:>13}: {value.data.item():.3f}")

`PeriodFreq` and `PeriodAmp` are NaN here: the trumpet's centroid does not repeat at a regular rate.

To do the same for a whole folder, `mir.batch` runs a set of features on every audio file. It replaces MIRtoolbox's `'Folder'` keyword. By default, each framed feature is summarised by its mean and standard deviation, as `mirexport` does. `mir.file_table` turns the result into a table with one row per file, and `mir.to_csv` saves it.

First we save the four clips as WAV files in a temporary folder:

In [ ]:
folder = Path(tempfile.mkdtemp())
for name in clips:
    y, clip_sr = librosa.load(librosa.ex(name), sr=sr, duration=30)
    sf.write(folder / f"{name}.wav", y, clip_sr)
print(sorted(path.name for path in folder.glob("*.wav")))

In [ ]:
features = {
    "rms": lambda x: mir.rms(x, frame=True),
    "lowenergy": mir.lowenergy,
    "centroid": lambda x: mir.centroid(x, frame=True),
    "flux": mir.flux,
}
rows = mir.batch(folder, features)
table = mir.file_table(rows)
mir.to_csv(table, folder / "features.csv")

df = pd.read_csv(folder / "features.csv")
df

This CSV file opens directly in R (`read.csv`), SPSS or JASP. A chart of two columns:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
clip_names = df["file"].str.removesuffix(".wav")
axes[0].bar(clip_names, df["RMS_Mean"])
axes[0].set_title("Average RMS")
axes[1].bar(clip_names, df["Spectralcentroid_Mean"])
axes[1].set_title("Average spectral centroid (Hz)")
plt.show()

For the curves of one file, frame by frame, use `summary=False` and `mir.frame_table`. All features in a frame table must share the same frames.

In [ ]:
trumpet_frames = mir.frame_table(
    {
        "rms": mir.rms(trumpet, frame=True),
        "centroid": mir.centroid(trumpet, frame=True),
    }
)
pd.DataFrame(trumpet_frames).head()

### ④ librosa

librosa has no batch function or summaries. A loop over the files and a pandas `DataFrame` do the job:

In [ ]:
librosa_rows = []
for path in sorted(folder.glob("*.wav")):
    y, _ = librosa.load(path, sr=None)
    librosa_rows.append(
        {
            "file": path.name,
            "rms_mean": np.mean(librosa.feature.rms(y=y)),
            "centroid_mean": np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
        }
    )
librosa_df = pd.DataFrame(librosa_rows)
librosa_df

### ⑤ Essentia

Essentia's `MusicExtractor` computes a large set of features, with their statistics, in one call per file. Its results are named like `lowlevel.spectral_centroid.mean`. It uses its own settings (for example, longer frames), not MIRtoolbox's.

In [ ]:
if es is None:
    print("Essentia is not available; skipped.")
else:
    extractor = es.MusicExtractor(lowlevelStats=["mean", "stdev"])
    essentia_rows = []
    for path in sorted(folder.glob("*.wav")):
        pool = extractor(str(path))[0]
        essentia_rows.append(
            {
                "file": path.name,
                "centroid_mean": pool["lowlevel.spectral_centroid.mean"],
            }
        )
    essentia_df = pd.DataFrame(essentia_rows)
    print(essentia_df)

### ⑥ Side by side

The average centroid of each clip from the three libraries:

In [ ]:
comparison = pd.DataFrame(
    {
        "mirpy": df["Spectralcentroid_Mean"].to_numpy(),
        "librosa": librosa_df["centroid_mean"].to_numpy(),
    },
    index=df["file"],
)
if es is not None:
    comparison["Essentia"] = essentia_df["centroid_mean"].to_numpy()

comparison.plot.bar(figsize=(8, 3), rot=0)
plt.ylabel("Average centroid (Hz)")
plt.show()

- All three find Vibe Ace and the Nutcracker the darkest clips.
- The trumpet and Brahms swap places. The trumpet clip has many near-silent frames between notes. mirpy leaves them out (as MIRtoolbox does), while librosa and Essentia average them in, and the faint noise in those frames has a high centroid.
- The averages also differ because each library uses its own frame length and spectrum settings.
- **mirpy** gives MIRtoolbox's summaries and column names, so existing MATLAB analyses can be redone. **librosa** leaves the summaries to you. **Essentia**'s `MusicExtractor` is the quickest way to get many features at once.

### Exercise

Add the brightness and the MFCCs to the batch, write the table to a CSV file, and read it back with pandas. Which clip has the highest average brightness? How many columns do the MFCCs add?

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
more_features = {
    "brightness": lambda x: mir.brightness(x, frame=True),
    "mfcc": lambda x: mir.mfcc(x, frame=True),
}
more = pd.DataFrame(mir.file_table(mir.batch(folder, more_features)))
print(more[["file", "Brightness_Mean"]])
print(sum(column.startswith("MFCC") for column in more.columns), "MFCC columns")

Each of the 13 coefficients gets a mean and a standard deviation: 26 columns.

## Summary

| MIRtoolbox | mirpy | librosa | Essentia |
|---|---|---|---|
| `mirmfcc(a, 'Frame')` | `mir.mfcc(a, frame=True)` | `feature.mfcc` (dB of power, rank 0 included) | `MFCC` (dB, rank 0 included) |
| `mirflux(a)` | `mir.flux(a)` | from `stft`; `onset.onset_strength` | `Flux` |
| `mirflux(a, 'SubBand')` | `mir.flux(a, sub_band="manual")` | – | – |
| `mirstat(f)` | `mir.stat(f)` | NumPy + pandas | `MusicExtractor` statistics |
| `mirrms('Folder')` + `mirexport` | `mir.batch` + `mir.file_table` + `mir.to_csv` | loop + pandas | `MusicExtractor` per file |

## Next

The next chapters, on pitch, tonality and rhythm, come with release 0.2.

## References

As cited in the MIRtoolbox 1.8.1 User's Manual.

- Lartillot, O., Toiviainen, P. "A Matlab Toolbox for Musical Feature Extraction From Audio", International Conference on Digital Audio Effects, Bordeaux, 2007. *MIRtoolbox.*
- Alluri, V. & Toiviainen, P. (2010). Exploring perceptual and acoustic correlates of polyphonic timbre. Music Perception, 27(3), 223–241. *Sub-band flux.*
- Alluri, V., Toiviainen, P., Jääskeläinen, I., Sams, M., Glerean, E., & Brattico, E. (2012). Large-scale brain networks emerge from dynamic processing of musical timbre, key an rhythm. Neuroimage 59, 3677-3689. *Sub-band flux.*